## mount and paths

In [1]:
from pathlib import Path
import sys

SEED = 42

# ============================================================
# DATA PATH
# Change this to your local dataset location
# ============================================================

DATA_DIR = Path(
    "/path/to/Collagen_Acetic_Acid"
)


# ============================================================
# PROJECT ROOT
# ============================================================

cwd = Path.cwd().resolve()

ROOT = None

for path in [cwd, *cwd.parents]:
    if (
        (path / "pyproject.toml").exists()
        and
        (path / "src" / "spectral_select").exists()
    ):
        ROOT = path
        break

if ROOT is None:
    raise FileNotFoundError(
        "Could not find the spectral-select repository root. "
        "Run the notebook from inside the cloned repository."
    )


# Allow imports from repo/src
SRC_DIR = ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


print("Project:", ROOT)
print("Data:", DATA_DIR)


Mounted at /content/drive
Project: /content/drive/MyDrive/Orbeli_ME-HSI_Selection
Data: /content/drive/MyDrive/Orbeli_ME-HSI_Selection/Data/processed/Collagen_Acetic_Acid


## data

In [2]:
from spectral_select.me_smartattention.data import load_dataset

data = load_dataset(
    DATA_DIR
)

Class 0: 3 specimens
Class 1: 3 specimens
Class 2: 3 specimens

Pixels: 46,670
Specimens: 9
Specimens/class: [3, 3, 3]
Ignored artifact pixels: 11
Classes: 3
Excitations: 6
Emission positions: 30
Valid measurements: 149
LR input: (46670, 149)
EEM input: (46670, 6, 30)


## splits

In [3]:
from spectral_select.me_smartattention.splits import (
    make_specimen_splits,
    make_pixel_splits,
    print_splits
)

specimen_splits = make_specimen_splits(
    data["X_flat"],
    data["y"],
    data["groups"],
    seed=SEED
)

pixel_splits = make_pixel_splits(
    data["y"],
    data["groups"],
    n_folds=3,
    seed=SEED
)

print_splits(
    specimen_splits,
    data["groups"],
    "Whole-Specimen Splits"
)

print_splits(
    pixel_splits,
    data["groups"],
    "Random Pixel Splits"
)


Whole-Specimen Splits
Fold 1: train specimens [1, 2, 4, 5, 7, 8] -> test specimens [0, 3, 6]
Fold 2: train specimens [0, 1, 2, 3, 6, 8] -> test specimens [4, 5, 7]
Fold 3: train specimens [0, 3, 4, 5, 6, 7] -> test specimens [1, 2, 8]

Random Pixel Splits
Fold 1: train specimens [0, 1, 2, 3, 4, 5, 6, 7, 8] -> test specimens [0, 1, 2, 3, 4, 5, 6, 7, 8]
Fold 2: train specimens [0, 1, 2, 3, 4, 5, 6, 7, 8] -> test specimens [0, 1, 2, 3, 4, 5, 6, 7, 8]
Fold 3: train specimens [0, 1, 2, 3, 4, 5, 6, 7, 8] -> test specimens [0, 1, 2, 3, 4, 5, 6, 7, 8]


## Baseline LR

In [4]:
from spectral_select.me_smartattention.baseline import run_logistic_regression

lr_specimen = run_logistic_regression(
    X=data["X_flat"],
    y=data["y"],
    groups=data["groups"],
    splits=specimen_splits,
    name="LR — Whole-Specimen Split",
    seed=SEED
)

lr_pixel = run_logistic_regression(
    X=data["X_flat"],
    y=data["y"],
    groups=data["groups"],
    splits=pixel_splits,
    name="LR — Random Pixel Split",
    seed=SEED
)


LR — Whole-Specimen Split
Fold 1: pixel 76.72% | F1 0.7602 | specimen 3/3 (100.00%)
Fold 2: pixel 65.63% | F1 0.4550 | specimen 2/3 (66.67%)
Fold 3: pixel 47.65% | F1 0.4266 | specimen 1/3 (33.33%)

Overall pixel accuracy: 63.61%
Macro F1: 0.6431
Specimen majority-vote accuracy: 6/9 (66.67%)

LR — Random Pixel Split
Fold 1: pixel 92.33% | F1 0.9235 | specimen 9/9 (100.00%)
Fold 2: pixel 92.97% | F1 0.9299 | specimen 9/9 (100.00%)
Fold 3: pixel 93.15% | F1 0.9318 | specimen 9/9 (100.00%)

Overall pixel accuracy: 92.82%
Macro F1: 0.9284
Specimen majority-vote accuracy: 9/9 (100.00%)


##our method

it seems for specimen heldout regime its best to increase regularization to not overfit which is logical. \
while for the random pixel split regularization decreases the score.

In [5]:
# ============================================================
# ME SMARTATTENTION — CONFIG + TRAIN
# ============================================================

from spectral_select.me_smartattention.training import run_smart_attention


# ---------------- MODEL CONFIG ----------------

DIM = 16
DEPTH = 3
HEADS = 2
DIM_HEAD = 8
MLP_DIM = 4

DROPOUT = 0.25
EMB_DROPOUT = 0.10

EDGE_MODE = "circular"
# ---------------- TRAINING CONFIG ----------------

EPOCHS = 30
BATCH_SIZE = 256
SAMPLES_PER_SPECIMEN = 2500

LR_RATE = 5e-4
WEIGHT_DECAY = 0.0

SCHEDULER_GAMMA = 0.9
SCHEDULER_STEPS = 10


# ---------------- COMMON PARAMETERS ----------------

config = dict(
    dim=DIM,
    depth=DEPTH,
    heads=HEADS,
    dim_head=DIM_HEAD,
    mlp_dim=MLP_DIM,

    dropout=DROPOUT,
    embedding_dropout=EMB_DROPOUT,

    edge_mode = EDGE_MODE,

    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    samples_per_specimen=SAMPLES_PER_SPECIMEN,

    lr=LR_RATE,
    weight_decay=WEIGHT_DECAY,

    scheduler_gamma=SCHEDULER_GAMMA,
    scheduler_steps=SCHEDULER_STEPS,

    seed=SEED
)


# ---------------- WHOLE-SPECIMEN SPLIT ----------------

smart_specimen = run_smart_attention(
    data=data,
    splits=specimen_splits,
    name="ME SmartAttention — Whole-Specimen Split",
    **config
)


# ---------------- RANDOM PIXEL SPLIT ----------------

smart_pixel = run_smart_attention(
    data=data,
    splits=pixel_splits,
    name="ME SmartAttention — Random Pixel Split",
    **config
)


ME SmartAttention — Whole-Specimen Split

Fold 1
Train specimens: [1, 2, 4, 5, 7, 8]
Test specimens: [0, 3, 6]

Model configuration
Parameters: 6,608
DIM: 16
Depth: 3
Heads: 2
Head dimension: 8
FFN dimension: 4
Dropout: 0.25
Embedding dropout: 0.1
Edge mode: circular
Epochs: 30
Learning rate: 0.0005
Weight decay: 0.0
Batch size: 256
Samples/specimen: 2500


Fold 1:   0%|          | 0/30 [00:00<?, ?it/s]

Fold 1: pixel 82.94% | F1 0.8245 | specimen 3/3 (100.00%)

Fold 2
Train specimens: [0, 1, 2, 3, 6, 8]
Test specimens: [4, 5, 7]


Fold 2:   0%|          | 0/30 [00:00<?, ?it/s]

Fold 2: pixel 88.49% | F1 0.5952 | specimen 3/3 (100.00%)

Fold 3
Train specimens: [0, 3, 4, 5, 6, 7]
Test specimens: [1, 2, 8]


...

Fold 1: pixel 98.12% | F1 0.9812 | specimen 9/9 (100.00%)

Fold 2
Train specimens: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Test specimens: [0, 1, 2, 3, 4, 5, 6, 7, 8]


Fold 2:   0%|          | 0/30 [00:00<?, ?it/s]

Fold 2: pixel 98.30% | F1 0.9830 | specimen 9/9 (100.00%)

Fold 3
Train specimens: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Test specimens: [0, 1, 2, 3, 4, 5, 6, 7, 8]


Fold 3:   0%|          | 0/30 [00:00<?, ?it/s]

Fold 3: pixel 98.28% | F1 0.9828 | specimen 9/9 (100.00%)

Overall pixel accuracy: 98.23%
Macro F1: 0.9823
Specimen majority-vote accuracy: 9/9 (100.00%)


In [10]:
# ============================================================
# ME SMARTATTENTION — CONFIG + TRAIN
# ============================================================

from spectral_select.me_smartattention.training import run_smart_attention


# ---------------- MODEL CONFIG ----------------

DIM = 16
DEPTH = 3
HEADS = 2
DIM_HEAD = 8
MLP_DIM = 4

DROPOUT = 0.25
EMB_DROPOUT = 0.10

EDGE_MODE = "zero"
# ---------------- TRAINING CONFIG ----------------

EPOCHS = 30
BATCH_SIZE = 256
SAMPLES_PER_SPECIMEN = 2500

LR_RATE = 5e-4
WEIGHT_DECAY = 0.0

SCHEDULER_GAMMA = 0.9
SCHEDULER_STEPS = 10


# ---------------- COMMON PARAMETERS ----------------

config = dict(
    dim=DIM,
    depth=DEPTH,
    heads=HEADS,
    dim_head=DIM_HEAD,
    mlp_dim=MLP_DIM,

    dropout=DROPOUT,
    embedding_dropout=EMB_DROPOUT,

    edge_mode = EDGE_MODE,

    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    samples_per_specimen=SAMPLES_PER_SPECIMEN,

    lr=LR_RATE,
    weight_decay=WEIGHT_DECAY,

    scheduler_gamma=SCHEDULER_GAMMA,
    scheduler_steps=SCHEDULER_STEPS,

    seed=SEED
)


# ---------------- WHOLE-SPECIMEN SPLIT ----------------

smart_specimen = run_smart_attention(
    data=data,
    splits=specimen_splits,
    name="ME SmartAttention — Whole-Specimen Split",
    **config
)


# ---------------- RANDOM PIXEL SPLIT ----------------

smart_pixel = run_smart_attention(
    data=data,
    splits=pixel_splits,
    name="ME SmartAttention — Random Pixel Split",
    **config
)


ME SmartAttention — Whole-Specimen Split

Fold 1
Train specimens: [1, 2, 4, 5, 7, 8]
Test specimens: [0, 3, 6]

Model configuration
Parameters: 6,608
DIM: 16
Depth: 3
Heads: 2
Head dimension: 8
FFN dimension: 4
Dropout: 0.25
Embedding dropout: 0.1
Edge mode: zero
Emission tokens: 30
Epochs: 30
Learning rate: 0.0005
Weight decay: 0.0
Batch size: 256
Samples/specimen: 2500


Fold 1:   0%|          | 0/30 [00:00<?, ?it/s]

Fold 1: pixel 82.65% | F1 0.8212 | specimen 3/3 (100.00%)

Fold 2
Train specimens: [0, 1, 2, 3, 6, 8]
Test specimens: [4, 5, 7]


Fold 2:   0%|          | 0/30 [00:00<?, ?it/s]

Fold 2: pixel 89.16% | F1 0.5994 | specimen 3/3 (100.00%)

Fold 3
Train specimens: [0, 3, 4, 5, 6, 7]
Test specimens: [1, 2, 8]


...

Fold 1: pixel 98.27% | F1 0.9826 | specimen 9/9 (100.00%)

Fold 2
Train specimens: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Test specimens: [0, 1, 2, 3, 4, 5, 6, 7, 8]


Fold 2:   0%|          | 0/30 [00:00<?, ?it/s]

Fold 2: pixel 98.23% | F1 0.9823 | specimen 9/9 (100.00%)

Fold 3
Train specimens: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Test specimens: [0, 1, 2, 3, 4, 5, 6, 7, 8]


Fold 3:   0%|          | 0/30 [00:00<?, ?it/s]

Fold 3: pixel 98.37% | F1 0.9837 | specimen 9/9 (100.00%)

Overall pixel accuracy: 98.29%
Macro F1: 0.9829
Specimen majority-vote accuracy: 9/9 (100.00%)


In [12]:
# ============================================================
# ME SMARTATTENTION — CONFIG + TRAIN
# ============================================================

from spectral_select.me_smartattention.training import run_smart_attention


# ---------------- MODEL CONFIG ----------------

DIM = 16
DEPTH = 3
HEADS = 2
DIM_HEAD = 8
MLP_DIM = 4

DROPOUT = 0.25
EMB_DROPOUT = 0.10

EDGE_MODE = "discard"
# ---------------- TRAINING CONFIG ----------------

EPOCHS = 30
BATCH_SIZE = 256
SAMPLES_PER_SPECIMEN = 2500

LR_RATE = 5e-4
WEIGHT_DECAY = 0.0

SCHEDULER_GAMMA = 0.9
SCHEDULER_STEPS = 10


# ---------------- COMMON PARAMETERS ----------------

config = dict(
    dim=DIM,
    depth=DEPTH,
    heads=HEADS,
    dim_head=DIM_HEAD,
    mlp_dim=MLP_DIM,

    dropout=DROPOUT,
    embedding_dropout=EMB_DROPOUT,

    edge_mode = EDGE_MODE,

    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    samples_per_specimen=SAMPLES_PER_SPECIMEN,

    lr=LR_RATE,
    weight_decay=WEIGHT_DECAY,

    scheduler_gamma=SCHEDULER_GAMMA,
    scheduler_steps=SCHEDULER_STEPS,

    seed=SEED
)


# ---------------- WHOLE-SPECIMEN SPLIT ----------------

smart_specimen = run_smart_attention(
    data=data,
    splits=specimen_splits,
    name="ME SmartAttention — Whole-Specimen Split",
    **config
)


# ---------------- RANDOM PIXEL SPLIT ----------------

smart_pixel = run_smart_attention(
    data=data,
    splits=pixel_splits,
    name="ME SmartAttention — Random Pixel Split",
    **config
)


ME SmartAttention — Whole-Specimen Split

Fold 1
Train specimens: [1, 2, 4, 5, 7, 8]
Test specimens: [0, 3, 6]

Model configuration
Parameters: 6,366
DIM: 16
Depth: 3
Heads: 2
Head dimension: 8
FFN dimension: 4
Dropout: 0.25
Embedding dropout: 0.1
Edge mode: discard
Emission tokens: 28
Epochs: 30
Learning rate: 0.0005
Weight decay: 0.0
Batch size: 256
Samples/specimen: 2500


Fold 1:   0%|          | 0/30 [00:00<?, ?it/s]

Fold 1: pixel 89.59% | F1 0.8963 | specimen 3/3 (100.00%)

Fold 2
Train specimens: [0, 1, 2, 3, 6, 8]
Test specimens: [4, 5, 7]


Fold 2:   0%|          | 0/30 [00:00<?, ?it/s]

Fold 2: pixel 90.89% | F1 0.6126 | specimen 3/3 (100.00%)

Fold 3
Train specimens: [0, 3, 4, 5, 6, 7]
Test specimens: [1, 2, 8]


...

Fold 1: pixel 98.20% | F1 0.9820 | specimen 9/9 (100.00%)

Fold 2
Train specimens: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Test specimens: [0, 1, 2, 3, 4, 5, 6, 7, 8]


Fold 2:   0%|          | 0/30 [00:00<?, ?it/s]

Fold 2: pixel 98.51% | F1 0.9851 | specimen 9/9 (100.00%)

Fold 3
Train specimens: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Test specimens: [0, 1, 2, 3, 4, 5, 6, 7, 8]


Fold 3:   0%|          | 0/30 [00:00<?, ?it/s]

Fold 3: pixel 98.15% | F1 0.9815 | specimen 9/9 (100.00%)

Overall pixel accuracy: 98.29%
Macro F1: 0.9829
Specimen majority-vote accuracy: 9/9 (100.00%)
